# **Notebook 5: Heat map**
> ### **Project: PestNeuroVision**
>
> **Description:** Visual explanation of the model's decision-making process. This notebook creates heat maps (**EigenCAM**) from the trained model generated in Notebook 1. The purpose is to identify the regions of interest in the input data that most influence the model's predictions.
>
> ---
> **Dependency:** Uses the trained model (`best.pt`) and specific samples from the test set.
>
> ---
>**License:** This notebook is licensed under the [GNU Affero General Public License v3.0 (AGPL-3.0)](https://www.gnu.org/licenses/agpl-3.0). See details in the [GitHub repository](https://github.com/alexander-lm/PestNeuroVision/tree/main).
>
>**Third-party software:** This project uses YOLO11 by [Ultralytics](https://github.com/ultralytics/ultralytics), distributed under the [GNU Affero General Public License v3.0](https://github.com/ultralytics/ultralytics/blob/main/LICENSE).

In [ ]:
!pip install grad-cam
!pip install Ultralytics

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import shutil
import os
import glob
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from IPython.display import Image, display
from ultralytics import settings, YOLO

In [ ]:
if os.path.exists("weights"):
    shutil.rmtree("weights")
    print("Folder weights deleted.")

if os.path.exists("inference_dataset"):
    shutil.rmtree("inference_dataset")
    print("Folder inference_dataset deleted.")

if os.path.exists("EigenCAM"):
    shutil.rmtree("EigenCAM")
    print("Folder EigenCAM deleted.")

os.makedirs('EigenCAM')
print("EigenCAM folder created.")

# Download the trained model (PesNeuroVision) and the test dataset from GitHub
!git init PestNeuroVision
%cd PestNeuroVision
!git remote add origin https://github.com/alexander-lm/PestNeuroVision.git
!git sparse-checkout init --cone
!git sparse-checkout set model-results/weights "dataset/inference_dataset_(CC BY 4.0)" "dataset/inference_dataset_(CC0 1.0)" image_credits.csv
!git pull --depth 1 origin main
!mv model-results/weights /content/weights
!mv "dataset/inference_dataset_(CC BY 4.0)" /content/inference_dataset
!find "dataset/inference_dataset_(CC0 1.0)" -type f -exec cp {} /content/inference_dataset/ \;
!mv image_credits.csv /content/image_credits.csv
%cd /content
!rm -rf PestNeuroVision



In [ ]:
# Trained model (PestNeuroVision)
yolo = YOLO('/content/weights/best.pt') # Trained model (PestNeuroVision)
output_dir = 'EigenCAM'

# Dataset
sources = sorted(glob.glob('/content/inference_dataset/*'))

# Heat map
class YOLOWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        output = self.model(x)
        if isinstance(output, (list, tuple)):
            return output[0]
        return output

torch_model = YOLOWrapper(yolo.model).eval()

target_layers = [
    yolo.model.model[6],
    yolo.model.model[8],
    yolo.model.model[10],
    yolo.model.model[16],
    yolo.model.model[19],
]

saved_images = []


plt.rcParams.update({
    'font.size':       18,
    'axes.titlesize':  22,
    'figure.titlesize': 26,
})

for i, source in enumerate(sources):
    img_bgr = cv2.imread(source)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (640, 640))
    tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
    tensor = tensor.unsqueeze(0)
    img_float = img_resized.astype(np.float32) / 255.0

    cam = EigenCAM(torch_model, target_layers)
    grayscale_cam = cam(input_tensor=tensor)[0]
    grayscale_inv = 1 - grayscale_cam

    overlay_normal = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)
    overlay_inv    = show_cam_on_image(img_float, grayscale_inv,  use_rgb=True)

    fig, axes = plt.subplots(2, 3, figsize=(20, 13))
    name = source.split('/')[-1].replace('.jpg', '')

    axes[0][0].imshow(img_resized);                  axes[0][0].set_title("Original");           axes[0][0].axis('off')
    axes[0][1].imshow(grayscale_cam, cmap='jet');    axes[0][1].set_title("EigenCAM");           axes[0][1].axis('off')
    axes[0][2].imshow(overlay_normal);               axes[0][2].set_title("Overlay");            axes[0][2].axis('off')
    axes[1][0].imshow(img_resized);                  axes[1][0].set_title("Original");           axes[1][0].axis('off')
    axes[1][1].imshow(grayscale_inv, cmap='jet');    axes[1][1].set_title("EigenCAM inverted");  axes[1][1].axis('off')
    axes[1][2].imshow(overlay_inv);                  axes[1][2].set_title("Overlay inverted");   axes[1][2].axis('off')

    fig.suptitle(name, fontsize=26, fontweight='bold')
    plt.tight_layout()

    save_path = f"{output_dir}/{name}.jpg"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    saved_images.append(save_path)
    print(f"Saved {i+1}/{len(sources)}: {name}.jpg")

# Note: You can see the heatmap images in the following folder: /content/EigenCAM

print("\nDisplaying all images:\n")
for path in saved_images:
    display(Image(filename=path, width=900))